# Advanced · C engine + classical search

How the bottom half of the framework works: the C engine that makes 1.4M place/undo cycles a second possible, the alpha-beta search that ships with it, the tactical and endgame algorithms that catch what MCTS would miss, and the inference infrastructure that lets dozens of workers share one GPU.

**Companion notebook:** `advanced_mcts_and_training.ipynb` covers MCTS internals, training math, replay priority, the AutoTuner, curriculum.

**Table of contents**

1. Bitboard representation
2. Incremental candidate set
3. Zobrist hashing
4. Alpha-beta search overview
5. Transposition table
6. Killer move heuristic
7. Late move reduction
8. Iterative deepening
9. Threat search algorithm
10. Endgame solver
11. Opening book (trie + Zobrist lookup)
12. ONNX export for self-play workers
13. GPU inference server
14. C MCTS (GIL-free, threaded)
15. Where to look in the source

In [ ]:
!pip install --quiet hexbot

## 1. Bitboard representation

Each player's stones live in a packed bitfield. The board is conceptually 19x19 = 361 cells, but the engine reserves a 32x32 region (1024 bits) so that horizontal shifts can detect lines without modular arithmetic.

**Win detection.** A 6-in-a-row along the q axis becomes: take the player's bitboard, shift it right by 1, AND with the original. The result is non-zero only where two adjacent stones exist. Repeat 5 times to get 6-in-a-row.

```
b1 = bitboard
b2 = b1 & (b1 >> 1)        // pairs
b4 = b2 & (b2 >> 2)        // quads
b6 = b4 & (b4 >> 2)        // sixes (4 + 2)
win = b6 != 0
```

Three line directions need three different shift offsets (for `(1,0)`, `(0,1)`, `(1,-1)`). The whole check is a handful of bitwise ops per direction. See `engine.c` `board_has_win`.

In [ ]:
# Measure win-detection throughput
from hexbot import HexGame
import time

g = HexGame()
for q, r in [(0,0), (1,0), (2,0), (3,0), (4,0), (5,0)]:
    pass  # we'll set up below

N = 100_000
t0 = time.perf_counter()
for _ in range(N):
    _ = g.is_over  # triggers C-side win check
dt = time.perf_counter() - t0
print(f"win-check throughput: {N/dt/1e6:.2f} M/sec ({dt:.3f}s for {N:,} calls)")

## 2. Incremental candidate set

The board is infinite, so 'all legal moves' is technically infinite too. The engine cheats: every cell within 2 rings of an existing stone is a candidate. `place(q, r)` adds new neighbours to the set; `undo()` removes them.

The set is stored as a small hash table (open addressing, no allocation per op). `legal_moves()` returns its contents directly. `scored_moves(n)` sorts on a per-cell heuristic score that the engine maintains incrementally too (line-extension potential, blocking value, proximity).

Net effect: search algorithms never enumerate the whole board. They get a handful (typically 30-80) of locally-relevant candidates per ply.

In [ ]:
g = HexGame()
print(f'empty board: {len(g.legal_moves())} candidates')
g.place(0, 0)
print(f'after 1 stone: {len(g.legal_moves())} candidates')
g.place(5, 5)
print(f'after 2 distant stones: {len(g.legal_moves())} candidates')

## 3. Zobrist hashing

Every position has a unique 64-bit hash that updates incrementally with each move:

$$h(s) = \bigoplus_{(p, q, r) \in \text{stones}} \text{ZOBRIST}[p][q][r]$$

where `ZOBRIST` is a precomputed table of random 64-bit numbers. XOR is its own inverse, so placing a stone is `h ^= ZOBRIST[p][q][r]` and undoing is the same operation.

Properties:

- **O(1) per move.** No re-hashing the whole board.
- **Transposition detection.** Two different move orders that reach the same position produce the same hash.
- **Cheap lookups.** The transposition table is just `tt[h & TT_MASK]`.

See `engine.c:70` (`ZOBRIST[2][SIZE][SIZE]`) and `init_zobrist`. The table is seeded with a fixed xorshift seed so hashes are deterministic across runs.

In [ ]:
# Confirm transposition: two move orders, same hash
g1 = HexGame()
g1.place(0, 0); g1.place(2, 0); g1.place(2, -1)

g2 = HexGame()
g2.place(0, 0); g2.place(2, -1); g2.place(2, 0)  # same stones, different order for P1

print(f'g1 zhash: {g1.zhash:016x}')
print(f'g2 zhash: {g2.zhash:016x}')
print(f'transposition: {g1.zhash == g2.zhash}')

## 4. Alpha-beta search overview

Standard alpha-beta with negamax framing. Pseudocode for one node:

```
search(node, depth, alpha, beta):
    if depth == 0 or terminal(node):
        return eval(node)
    for move in ordered_moves(node):
        node.place(move)
        score = -search(node, depth-1, -beta, -alpha)
        node.undo()
        if score >= beta:
            record_killer(depth, move)
            return beta              # beta cutoff
        alpha = max(alpha, score)
    return alpha
```

The optimisations below all sit inside `ordered_moves` and the cutoff path. See `engine.c` `negamax_with_tt`.

In [ ]:
g = HexGame()
g.place(0, 0)
g.place(2, 0); g.place(2, -1)

for depth in [2, 4, 6, 8]:
    r = g.search(depth=depth)
    print(f"depth={depth}: best={r['best_move']}  value={r['value']:+.2f}  nodes={r['nodes']:>7,}")

## 5. Transposition table

After each search at a node, the engine stores `{zobrist_hash, depth, value, best_move, flag}` in a global table:

$$\text{tt}[h \bmod 2^{23}] = (h, d, v, m, \text{flag})$$

Size: `TT_SIZE = 1 << 23` (about 8 million entries, ~384 MB). `TT_MASK` is the bitwise modulo. The flag distinguishes exact / lower-bound / upper-bound entries so the search can use the cached value correctly for the current `(alpha, beta)` window.

On entry to a node:

- TT hit, sufficient depth, exact: return immediately.
- TT hit, sufficient depth, bound that proves cutoff: return.
- TT hit, any depth: use TT's best_move as the first candidate to try (move ordering).
- TT miss: search normally.

Defined at `engine.c:639` (`TT_SIZE`, `TT_MASK`, `tt_table`).

## 6. Killer move heuristic

Move ordering matters enormously in alpha-beta. The closer to the front you try good moves, the more cutoffs you get, and `b^d` becomes `b^(d/2)` in the best case.

**Killers** are moves that produced a beta cutoff at the same ply earlier in the search. They are likely to produce a cutoff in a sibling subtree too, even when the board is slightly different. The engine keeps two killer slots per ply (`engine.c:670`):

```c
static int killer_q[MAX_PLY][2];
static int killer_r[MAX_PLY][2];
```

Order at each node:

1. TT best move (if any).
2. Killers for this ply (most recent first).
3. C heuristic score, descending.

When a move produces a cutoff, it gets promoted into slot 0; the previous slot 0 is demoted to slot 1; slot 1 is dropped.

## 7. Late move reduction

Most moves in alpha-beta are bad and produce no cutoff. Searching every one to full depth is wasteful. Late move reduction (LMR) searches the tail of the move list at reduced depth first. If the reduced search returns a value high enough to potentially raise alpha, re-search at full depth.

Heuristic (in `engine.c`):

- First few moves at each node: search at full depth.
- Remaining moves: search at depth - 1 (or depth - 2 deeper in the tree).
- If a reduced search threatens to raise alpha, re-search at full depth.

This is configurable via compile-time defines in `engine.c` (search for `LMR_`). Default settings give ~30% node reduction at depth 8 with no measurable strength loss.

## 8. Iterative deepening

Rather than searching directly at depth N, alpha-beta searches at depth 1, then 2, then 3, ..., then N. This seems wasteful (most nodes are searched multiple times) but it pays off because:

- The transposition table built at depth N-1 dramatically improves move ordering at depth N.
- You always have a usable answer even if a deep search runs out of time.
- Aspiration windows (search at depth N with a narrow `(alpha, beta)` band centred on the depth N-1 value) get further cutoffs.

`game.search(depth=8)` does iterative deepening internally. The reported `nodes` is the total across all depths.

## 9. Threat search algorithm

Pure alpha-beta has to explore the whole branching factor at every ply. **Threat search** restricts exploration to moves that are either creating a threat or responding to one. This gives near-tactical-depth at a tiny fraction of the cost.

Pseudocode (`orca/threats.py`):

```
threat_search(node, depth, side_to_threaten):
    threats = find_threats(node, side_to_threaten)
    if not threats:
        return None
    for t in threats:
        node.place(t)
        defenses = find_winning_moves(node, opponent(side))
        if not defenses:                       // unstoppable
            node.undo(); return t
        if depth > 1:
            for d in defenses:
                node.place(d)
                follow_up = threat_search(node, depth-1, side)
                node.undo()
                if follow_up:
                    node.undo(); return t
        node.undo()
    return None
```

Returns the threat-creating move if a forced win exists within `depth` threat-pairs. The bot calls `threat_search(game, depth=4)` before MCTS as a tactical shortcut.

In [ ]:
from hexbot import threat_search

g = HexGame()
g.place(10, 10)
g.place(0, 0); g.place(1, 0)
g.place(11, 11); g.place(12, 12)
g.place(2, 0); g.place(3, 0)        # P1 four in a row
g.place(13, 13); g.place(14, 14)
g.place(4, 0)                       # P1 makes 5

tactical = threat_search(g, depth=4)
print(f'threat_search depth 4: {tactical}')

## 10. Endgame solver

When the position is tactically sharp and the candidate set is small, brute-force is feasible. `orca/solver.py` wraps the C engine's alpha-beta at high depth with a time budget, and reads the transposition table back to give an exact result:

- `result['result']`: one of `'win'`, `'loss'`, `'draw'`, `'unknown'` (timeout).
- `result['move']`: the move that achieves the result.
- `result['depth_reached']`: how many plies the iterative deepening managed.
- `result['nodes']`: how many positions were evaluated.

Internally it shares the same TT, killers, and LMR as `game.search()`, just with a much higher depth budget.

In [ ]:
from hexbot import solve

g = HexGame()
g.place(0, 0)
g.place(0, 5); g.place(0, 6)
g.place(1, 0); g.place(2, 0)
g.place(1, 5); g.place(1, 6)
g.place(3, 0); g.place(4, 0)
g.place(2, 5); g.place(2, 6)

r = solve(g, max_depth=6, time_limit=2.0)
print(f"result: {r['result']}")
print(f"move:   {r['move']}")

## 11. Opening book (trie + Zobrist lookup)

Two lookup paths in `orca/openings.py`:

**Zobrist hash table** (O(1) lookup): `{zhash: {move: frequency}}`. The book is built by scanning winning games and storing positions reached. At inference, you compute the current position's zhash and look it up directly.

**Move-sequence trie** (fallback): `{move_tuple: {next_move: frequency}}`. Used when the Zobrist table misses (typically when the position deviates from the book set). The trie matches by exact move sequence rather than position.

The book is built once from a corpus of high-quality games and serialised to disk. At inference, the bot consults the book before MCTS for the first ~10 moves; the move with highest frequency that also passes a temperature-weighted noise check is returned.

In [ ]:
from orca.openings import OpeningBook, build_default_book

book = build_default_book()
print(f'opening book size: {len(book)} positions')

## 12. ONNX export for self-play workers

Self-play happens in process pool workers (one per core). Loading a fresh PyTorch model in every worker every iteration would dominate the cost. The trainer's solution:

1. At the start of each iteration, `bot.export_onnx(net, '/tmp/hex_model_<iter>.onnx')` serialises the model in ONNX format.
2. Workers `import onnxruntime; sess = ort.InferenceSession(path)` on startup.
3. Per-move inference: `sess.run(['policy', 'value', 'threat'], {'input': encoded_state})`.

Why ONNX:

- **CPU efficiency.** ONNX Runtime is 3-5x faster than PyTorch on CPU for these forward passes.
- **No PyTorch import.** Workers don't need to import torch at all (slow startup).
- **Stable serialization format.** Survives PyTorch version upgrades.

See `bot.py` `export_onnx` and `_self_play_worker_v1` in `orca/train.py`.

## 13. GPU inference server

On CUDA hosts, ONNX-on-CPU underutilises the GPU. The GPU inference server pattern (`orca/gpu_server.py`) reverses the data flow:

```
Workers (CPU game logic) -> Queue -> GPU Server (batched NN) -> Queue -> Workers
```

- One server process keeps the network resident on GPU.
- Many worker threads/processes generate game positions on CPU using the C engine.
- Workers send positions to the server queue and block on the response queue.
- The server batches up to N positions, runs one GPU forward, and dispatches results back.

Latency per position is higher (queue + batch wait) but throughput is several times better than CPU-bound ONNX inference. Used during training on CUDA when worker count > 4.

## 14. C MCTS (GIL-free, threaded)

Standard `BatchedMCTS` is Python and holds the GIL while building the tree. For very high throughput on multi-GPU CUDA hosts, `orca/c_mcts.py` exposes a C implementation that:

- Builds the tree entirely in C (`mcts_tree_new`, `mcts_select_batch`, `mcts_expand_backup`).
- Releases the GIL during tree work.
- Returns batches of leaf encodings to Python, which runs them through the network, then passes results back to C for backup.

Net effect: multiple Python threads can each drive a C MCTS tree concurrently against a shared GPU inference server, without GIL contention. See `c_mcts.search()` in `orca/c_mcts.py` and the corresponding `mcts_*` functions in `engine.c`.

## 15. Where to look in the source

Map from the topics in this notebook to actual files and line numbers:

| Topic | File | Anchor |
|---|---|---|
| Bitboard structures | `engine.c` | `Board` struct + `board_has_win` |
| Candidate set | `engine.c` | `candidates_add` / `candidates_remove` |
| Zobrist table | `engine.c:70` | `ZOBRIST[2][SIZE][SIZE]` |
| Alpha-beta core | `engine.c` | `negamax_with_tt`, line ~688 |
| Transposition table | `engine.c:639` | `TT_SIZE`, `tt_table[TT_SIZE]` |
| Killer moves | `engine.c:670` | `killer_q`, `killer_r`, `record_killer` |
| Threat search | `orca/threats.py` | `_threat_search`, `find_forced_move` |
| Endgame solver | `orca/solver.py` | `solve`, `solver_or_mcts` |
| Opening book | `orca/openings.py:35` | `class OpeningBook` |
| ONNX export | `bot.py` | `export_onnx` |
| GPU inference server | `orca/gpu_server.py` | top-of-file docstring |
| C MCTS | `orca/c_mcts.py`, `engine.c` | `mcts_tree_new`, `mcts_select_batch` |

Sister notebook: `advanced_mcts_and_training.ipynb` for MCTS math, training loss, replay priority, AutoTuner, curriculum, plateau detection.